# 🚀 XGBoost v6 Training - Pure Engine

**Strategy:**
- Train on balanced v6 engineered data
- NO monotonic constraints
- Let model learn naturally from clean data

**Output:** `xgboost_v6_pure_final.json`

## 1. Setup

In [ ]:
!pip install polars xgboost -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
import os

print(f"Polars: {pl.__version__}")
print(f"XGBoost: {xgb.__version__}")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data/processed"

FILES = {
    "train": f"{BASE_PATH}/master_train_v6_engineered.parquet",
    "test": f"{BASE_PATH}/master_test_v6_engineered.parquet",
    "model_out": f"{BASE_PATH}/xgboost_v6_pure_final.json"
}

for name, path in FILES.items():
    if "_out" not in name:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"{exists} {name}: {path}")

## 2. Load v6 Engineered Data

In [ ]:
# Features for model
FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]
TARGET = 'bolus'

LOAD_COLS = FEATURES + [TARGET]
print(f"📋 Loading {len(LOAD_COLS)} columns")

In [ ]:
print("📂 Loading TRAIN...")
df_train = pl.read_parquet(FILES["train"], columns=LOAD_COLS)
print(f"   Rows: {len(df_train):,}")

print("\n📂 Loading TEST...")
df_test = pl.read_parquet(FILES["test"], columns=LOAD_COLS)
print(f"   Rows: {len(df_test):,}")

In [ ]:
# Fill any remaining nulls
for col in FEATURES:
    if col in df_train.columns:
        df_train = df_train.with_columns(pl.col(col).fill_null(0))
        df_test = df_test.with_columns(pl.col(col).fill_null(0))

print("✅ Nulls filled")

## 3. Prepare XGBoost Data

In [ ]:
# Get available features
available_features = [f for f in FEATURES if f in df_train.columns]
print(f"📋 Available features: {len(available_features)}")
print(f"   {available_features}")

In [ ]:
# Extract arrays
print("\n📐 Extracting arrays...")

X_train = df_train.select(available_features).to_numpy().astype(np.float32)
y_train = df_train[TARGET].to_numpy().astype(np.float32)

X_test = df_test.select(available_features).to_numpy().astype(np.float32)
y_test = df_test[TARGET].to_numpy().astype(np.float32)

print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")

In [ ]:
# Create DMatrix
print("\n🔧 Creating DMatrix...")

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=available_features)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=available_features)

print("✅ DMatrix created")

## 4. XGBoost Configuration (GPU, NO Constraints)

In [ ]:
params = {
    # Objective & Metric
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    
    # GPU Mode
    'tree_method': 'hist',
    'device': 'cuda',
    
    # Hyperparameters
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    
    # Regularization
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    
    # NO MONOTONIC CONSTRAINTS!
    # 'monotone_constraints': None  # Explicitly disabled
    
    # Reproducibility
    'seed': 42
}

print("⚙️ XGBoost Parameters:")
for k, v in params.items():
    print(f"   {k}: {v}")

print("\n⚠️ MONOTONIC CONSTRAINTS: DISABLED")

## 5. Training

In [ ]:
# Training parameters
NUM_ROUNDS = 1000
EARLY_STOPPING = 50

print(f"🚀 Training XGBoost v6 Pure Engine...")
print(f"   Rounds: {NUM_ROUNDS}")
print(f"   Early Stopping: {EARLY_STOPPING}")
print("=" * 50)

In [ ]:
# Train!
evals = [(dtrain, 'train'), (dtest, 'test')]

model = xgb.train(
    params,
    dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=50  # Print every 50 rounds
)

print("\n✅ Training complete!")

In [ ]:
# Best iteration
print(f"\n📊 TRAINING RESULTS:")
print(f"   Best Iteration: {model.best_iteration}")
print(f"   Best Score: {model.best_score:.4f}")

## 6. Evaluation

In [ ]:
# Predictions
print("🔮 Generating predictions...")

preds_train = model.predict(dtrain)
preds_test = model.predict(dtest)

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

train_mae = mean_absolute_error(y_train, preds_train)
test_mae = mean_absolute_error(y_test, preds_test)

train_rmse = np.sqrt(mean_squared_error(y_train, preds_train))
test_rmse = np.sqrt(mean_squared_error(y_test, preds_test))

print(f"\n📊 EVALUATION METRICS:")
print(f"   Train MAE:  {train_mae:.4f}")
print(f"   Test MAE:   {test_mae:.4f}")
print(f"   Train RMSE: {train_rmse:.4f}")
print(f"   Test RMSE:  {test_rmse:.4f}")

In [ ]:
# Positive-only metrics (clinical relevance)
pos_mask_train = y_train > 0
pos_mask_test = y_test > 0

if pos_mask_train.sum() > 0:
    train_pos_mae = mean_absolute_error(y_train[pos_mask_train], preds_train[pos_mask_train])
else:
    train_pos_mae = 0

if pos_mask_test.sum() > 0:
    test_pos_mae = mean_absolute_error(y_test[pos_mask_test], preds_test[pos_mask_test])
else:
    test_pos_mae = 0

print(f"\n📊 POSITIVE-ONLY METRICS (Clinically Relevant):")
print(f"   Train Positive MAE: {train_pos_mae:.4f}")
print(f"   Test Positive MAE:  {test_pos_mae:.4f}")

## 7. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

# Get importance
importance = model.get_score(importance_type='gain')
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)

print("📊 FEATURE IMPORTANCE (Gain):")
for feat, score in importance_sorted:
    print(f"   {feat:25} {score:.2f}")

# Plot
plt.figure(figsize=(10, 6))
xgb.plot_importance(model, importance_type='gain', max_num_features=12)
plt.title('Feature Importance (Gain)')
plt.tight_layout()
plt.show()

## 8. Clinical Safety Checks

In [ ]:
print("🏥 CLINICAL SAFETY CHECKS")
print("=" * 50)

# Test 1: Hypo scenario (CGM=60, no carbs)
test_hypo = np.zeros((1, len(available_features)), dtype=np.float32)
test_hypo[0, available_features.index('CGM')] = 60.0
test_hypo[0, available_features.index('carbs')] = 0.0

d_hypo = xgb.DMatrix(test_hypo, feature_names=available_features)
pred_hypo = model.predict(d_hypo)[0]

status = "✅ SAFE" if pred_hypo < 0.1 else "⚠️ DANGEROUS"
print(f"\n1. HYPO TEST (CGM=60, Carbs=0):")
print(f"   Prediction: {pred_hypo:.4f} U")
print(f"   Status: {status}")

# Test 2: High glucose with carbs
test_high = np.zeros((1, len(available_features)), dtype=np.float32)
test_high[0, available_features.index('CGM')] = 250.0
test_high[0, available_features.index('carbs')] = 60.0

d_high = xgb.DMatrix(test_high, feature_names=available_features)
pred_high = model.predict(d_high)[0]

status = "✅ GOOD" if pred_high > 1.0 else "⚠️ TOO LOW"
print(f"\n2. HIGH GLUCOSE TEST (CGM=250, Carbs=60):")
print(f"   Prediction: {pred_high:.4f} U")
print(f"   Status: {status}")

# Test 3: Normal glucose, no food
test_normal = np.zeros((1, len(available_features)), dtype=np.float32)
test_normal[0, available_features.index('CGM')] = 120.0
test_normal[0, available_features.index('carbs')] = 0.0

d_normal = xgb.DMatrix(test_normal, feature_names=available_features)
pred_normal = model.predict(d_normal)[0]

status = "✅ SAFE" if pred_normal < 0.5 else "⚠️ CHECK"
print(f"\n3. NORMAL TEST (CGM=120, Carbs=0):")
print(f"   Prediction: {pred_normal:.4f} U")
print(f"   Status: {status}")

## 9. Save Model

In [ ]:
print(f"\n💾 Saving model: {FILES['model_out']}")
model.save_model(FILES['model_out'])
print("✅ Model saved!")

In [ ]:
print("\n" + "=" * 70)
print("📋 TRAINING SUMMARY")
print("=" * 70)
print(f"""
MODEL: XGBoost v6 Pure Engine
   Monotonic Constraints: DISABLED
   Best Iteration: {model.best_iteration}

DATA:
   Train: {len(df_train):,} rows (Balanced 50/50)
   Test: {len(df_test):,} rows (Real-world)

METRICS:
   Test MAE: {test_mae:.4f}
   Test RMSE: {test_rmse:.4f}
   Positive-Only MAE: {test_pos_mae:.4f}

OUTPUT:
   ✅ {FILES['model_out']}
""")
print("=" * 70)
print("🏆 TRAINING COMPLETE!")